# Project 6 — Part 3: LSTM Modeling of Rossmann Sales (`04_LSTM_Modeling.ipynb`)

**Objective.** Build a small LSTM (maximum two LSTM layers, CPU-friendly) that predicts the next value of the chain-wide series

> **y(t) = average sales per trading store on day t**

and evaluate it fairly against the naive benchmarks defined in `03_Time_Series_Analysis.ipynb`.

This notebook is **self-contained**: it reads only the raw `train.csv` / `store.csv` (or `rossmann-store-sales.zip`), never writes or modifies any file, and needs no `src/` folder or saved model.

## 2. Project requirements and forecasting approach

**Requirements (PDF item 2.6):** isolate the time series → check stationarity → difference if needed → ACF/PACF → sliding-window supervised data → scale to (−1, 1) → LSTM regression for the *next* value; model not deeper than **two LSTM layers**; must run on a normal CPU / Colab.

**Decisions inherited from `03_Time_Series_Analysis.ipynb`** (all taken on the training part only):

| Topic | Decision |
|---|---|
| Target | y(t) = mean `Sales` over *trading* stores (`Open == 1` and `Sales > 0`) per calendar day |
| Hold-out | the last **42 days** (the forecast horizon of the business case: six weeks) |
| Stationarity | ADF rejects a unit root but KPSS rejects stationarity → **no differencing**; the undifferenced series is fed to the LSTM |
| Window | **28 days** (4 weeks, two promotion cycles) is primary; 42 days only as a sensitivity check |
| Scaling | `MinMaxScaler(-1, 1)`, fitted on training data only |
| Naive benchmarks | *one-step*: same weekday 7 / 14 / 28 days ago, mean of lags 14 & 28, mean of lags 7-14-21-28 (all use actual history). *42 days ahead from one origin*: repeat the last 7 / 14 / 28 days |
| Metric | RMSPE = √mean(((actual − pred)/actual)²) |

**Two separate evaluation modes (never mixed):**
* **A. Rolling one-step-ahead** — to predict day *t* the model sees the **actual** values of the 28 days before *t*, including actual hold-out days that have already passed. This is exactly how the one-step naive forecasts work (`y.shift(k)`).
* **B. Fixed-origin 42-day forecast** — the origin is the last training day. Only data up to the origin is used; after that the model feeds on **its own predictions** (recursive forecasting). No hold-out actual is ever fed back. This matches the "repeat the last block" naive forecasts.

**Planned architecture:** input (28, 1) → `LSTM(32)` → `Dense(1)`; one LSTM layer, no dropout, Adam, MSE loss, batch 32, at most 100 epochs, early stopping on validation loss (patience 12, best weights restored).

## 3. Imports and reproducibility settings

In [1]:
import os, random, time, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED); np.random.seed(SEED)

try:
    import tensorflow as tf
except ImportError as e:      # do not silently substitute another model
    raise ImportError(f"TensorFlow/Keras is required for this notebook but could not be imported: {e}") from e

tf.keras.utils.set_random_seed(SEED)               # seeds Python, NumPy and TensorFlow together
try:
    tf.config.experimental.enable_op_determinism() # makes CPU ops deterministic where supported
except Exception as e:
    print("Op determinism not available:", e)

BLUE, ORANGE, AQUA, GREY, INK, RED = "#2a78d6", "#eb6834", "#1baf7a", "#8a8a85", "#0b0b0b", "#c0392b"
plt.rcParams.update({"figure.dpi": 100, "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True,
                     "grid.alpha": .25, "axes.titlesize": 12, "axes.titleweight": "bold", "axes.labelsize": 10})
pd.set_option("display.width", 170); pd.set_option("display.max_columns", 40)

# ---- Configuration: fixed BEFORE any model is trained or the hold-out is looked at ----
HOLDOUT_DAYS = 42      # same final hold-out as 03_Time_Series_Analysis.ipynb
VAL_DAYS     = 56      # validation = last 8 weeks of the training part
WINDOW       = 28      # primary input window (decided in the Time Series notebook)
WINDOW_SENS  = 42      # sensitivity experiment only
LSTM_UNITS   = 32
DROPOUT      = 0.0     # no dropout: tiny model, see section 10
BATCH_SIZE   = 32
MAX_EPOCHS   = 100
PATIENCE     = 12
LEARNING_RATE = 1e-3

print(f"TensorFlow {tf.__version__} | NumPy {np.__version__} | pandas {pd.__version__} | seed {SEED}")

ModuleNotFoundError: No module named 'sklearn'

## 4. Data loading and validation
The raw files are searched in the current folder, `data/raw/` and the parent folder; if `train.csv` is not found as a file it is read **directly from `rossmann-store-sales.zip`** (nothing is extracted or written). A missing file stops the notebook with its exact filename.

In [ ]:
SEARCH_DIRS = [Path.cwd(), Path.cwd() / "data" / "raw", Path.cwd().parent / "data" / "raw", Path.cwd().parent]

def locate(name):
    for d in SEARCH_DIRS:
        if (d / name).exists():
            return d / name
    return None

def read_raw(name, **kw):
    p = locate(name)
    if p is not None:
        print(f"{name}: read from {p}")
        return pd.read_csv(p, **kw)
    z = locate("rossmann-store-sales.zip")
    if z is not None:
        with zipfile.ZipFile(z) as zf:
            if name in zf.namelist():
                print(f"{name}: read from inside {z}")
                with zf.open(name) as f:
                    return pd.read_csv(f, **kw)
    raise FileNotFoundError(f"Required file '{name}' not found (looked in {[str(d) for d in SEARCH_DIRS]} and rossmann-store-sales.zip).")

train = read_raw("train.csv", usecols=["Store", "DayOfWeek", "Date", "Sales", "Open"], parse_dates=["Date"],
                 dtype={"Store": "int16", "DayOfWeek": "int8", "Sales": "int32", "Open": "int8"})
store = read_raw("store.csv", usecols=["Store", "StoreType", "Assortment"])
print("train:", train.shape, "| store:", store.shape)
print("Date dtype:", train.Date.dtype, "| raw file already ascending:", train.Date.is_monotonic_increasing)

# Same treatment as the Time Series notebook: explicit chronological sort, integrity checks
train = train.sort_values(["Date", "Store"]).reset_index(drop=True)
assert train.Date.is_monotonic_increasing
assert (train.Date.dt.dayofweek + 1 == train.DayOfWeek).all(), "DayOfWeek does not match Date"
assert not train.duplicated(["Store", "Date"]).any(), "duplicate (Store, Date)"
assert set(train.Store.unique()) <= set(store.Store), "train contains stores missing from store.csv"
print("After sorting: ascending =", train.Date.is_monotonic_increasing, "| first/last date:", train.Date.min().date(), train.Date.max().date())

## 5. Target-series construction (exactly as in the Time Series notebook)
* **Trading store** = `Open == 1` **and** `Sales > 0`. Closed stores (and open rows with zero sales) are dropped *before* averaging, so closures do not pull the average down.
* Group by `Date`, take the mean of `Sales` → y(t); `asfreq("D")` creates a regular daily index; any gap would become NaN and stop the notebook (the Time Series notebook found none).

In [ ]:
trading = (train.Open == 1) & (train.Sales > 0)
daily = train[trading].groupby("Date").agg(mean_sales=("Sales", "mean"), n_trading=("Store", "size")).asfreq("D")
n_missing_days = int(daily.mean_sales.isna().sum())
assert n_missing_days == 0, "the calendar has missing days"
y = daily.mean_sales.astype("float64")
y.name = "avg_sales_per_trading_store"

print(f"{len(y)} daily observations | {y.index.min().date()} -> {y.index.max().date()} | freq {y.index.freq} | missing days: {n_missing_days}")
print(f"Rows dropped as non-trading: {(~trading).sum():,} of {len(train):,}")
display(y.describe().round(1).to_frame("y(t), EUR per trading store"))

## 6. Date range and chronological split verification
```
|<------------- training part (900 days) ------------->|<-- hold-out (42 days) -->|
|<--- fit: train LSTM + fit scaler --->|<- validation (56) ->|   final test, used once
```
* **Fit part** – trains the LSTM and fits the scaler.
* **Validation (last 56 days of the training part)** – early stopping and model comparison.
* **Hold-out (last 42 days)** – identical to the Time Series notebook; touched only for the final scoring.

In [ ]:
last_day = y.index.max()
cut = last_day - pd.Timedelta(days=HOLDOUT_DAYS - 1)          # first hold-out day (same formula as the TS notebook)
y_trainpart = y[y.index < cut]                                # 'training part' of the TS notebook
y_ho  = y[y.index >= cut]                                     # final hold-out
val_start = y_trainpart.index[-VAL_DAYS]
y_fit = y_trainpart[y_trainpart.index < val_start]
y_val = y_trainpart[y_trainpart.index >= val_start]
origin = cut - pd.Timedelta(days=1)                           # forecast origin = last training day

split = pd.DataFrame({"part": ["fit (train)", "validation", "training part (fit+val)", "hold-out", "all"],
                      "start": [y_fit.index[0], y_val.index[0], y_trainpart.index[0], y_ho.index[0], y.index[0]],
                      "end":   [y_fit.index[-1], y_val.index[-1], y_trainpart.index[-1], y_ho.index[-1], y.index[-1]],
                      "days":  [len(y_fit), len(y_val), len(y_trainpart), len(y_ho), len(y)]})
split["start"] = split.start.dt.date; split["end"] = split.end.dt.date
display(split)
print("Forecast origin (last training day):", origin.date(), "| forecast horizon:", HOLDOUT_DAYS, "days")

# Values recorded in 03_Time_Series_Analysis.ipynb: 942 days, 2013-01-01 -> 2015-07-31, training 900 days (to 2015-06-19), hold-out 42 days (2015-06-20 -> 2015-07-31)
checks = []
def check(name, cond):
    cond = bool(cond); checks.append({"check": name, "passed": cond})
    assert cond, f"CHECK FAILED: {name}"

check("series index is strictly increasing, daily, no gaps", y.index.is_monotonic_increasing and y.index.is_unique and (np.diff(y.index.values) == np.timedelta64(1, "D")).all())
check("942 observations (as in Time Series notebook)", len(y) == 942)
check("date range 2013-01-01 -> 2015-07-31", y.index[0] == pd.Timestamp("2013-01-01") and y.index[-1] == pd.Timestamp("2015-07-31"))
check("training part = 900 days ending 2015-06-19", len(y_trainpart) == 900 and y_trainpart.index[-1] == pd.Timestamp("2015-06-19"))
check("hold-out = exactly 42 days, 2015-06-20 -> 2015-07-31", len(y_ho) == 42 and y_ho.index[0] == pd.Timestamp("2015-06-20") and y_ho.index[-1] == pd.Timestamp("2015-07-31"))
check("fit + validation + hold-out sizes add up to 942", len(y_fit) + len(y_val) + len(y_ho) == len(y))
check("fit < validation < hold-out in time (no overlap)", y_fit.index[-1] < y_val.index[0] and y_val.index[-1] < y_ho.index[0])
check("target values are strictly positive (needed for RMSPE)", (y > 0).all())
print("Chronology / boundary checks passed.")

## 7. Target-series visualization

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(y_fit.index, y_fit, color=BLUE, lw=.9, label=f"fit ({len(y_fit)} d)")
ax.plot(y_val.index, y_val, color=AQUA, lw=1.1, label=f"validation ({len(y_val)} d)")
ax.plot(y_ho.index, y_ho, color=ORANGE, lw=1.4, label=f"hold-out ({len(y_ho)} d)")
ax.axvline(val_start, color=GREY, ls=":", lw=1); ax.axvline(cut, color=GREY, ls=":", lw=1)
ax.set_ylabel("EUR per trading store"); ax.set_title("y(t): average sales per trading store, chronological split")
ax.legend(frameon=False, ncol=3, loc="upper left"); plt.tight_layout(); plt.show()

## 7b. Stationarity and ACF / PACF (PDF item 2.6, steps 2–4)
These three steps were carried out in `03_Time_Series_Analysis.ipynb` (sections 8–9) on the same 900-day training part, with this recorded outcome:

| Step | Result recorded in the Time Series notebook |
|---|---|
| Stationarity | raw y(t): ADF rejects a unit root (stat −5.27) but KPSS rejects stationarity (stat 1.10) → *conflicting*; around a linear trend both tests agree on stationary |
| Differencing | not applied by the pre-set rule (ADF p ≤ 0.01); differenced series are stationary, but differencing removes level information, so the LSTM uses the **undifferenced** series |
| ACF / PACF | strongest ACF lags 14, 28, 1, 42 → a 28-day window covers two promotion cycles |

For completeness, the ACF/PACF are re-drawn below with plain NumPy (no extra package) on the **training part only (hold-out excluded)**, so this notebook visibly supports the 28-day window choice. The ADF/KPSS tests are *not* repeated here; see the Time Series notebook.

In [ ]:
def acf_np(x, nlags):
    x = np.asarray(x, float) - np.mean(x); n = len(x); c0 = np.sum(x * x) / n
    return np.array([1.0] + [np.sum(x[:-k] * x[k:]) / n / c0 for k in range(1, nlags + 1)])

def pacf_np(x, nlags):                      # Yule-Walker via Durbin-Levinson recursion
    r = acf_np(x, nlags); phi = np.zeros((nlags + 1, nlags + 1)); out = [1.0]
    phi[1, 1] = r[1]; out.append(r[1])
    for k in range(2, nlags + 1):
        num = r[k] - np.sum(phi[k - 1, 1:k] * r[k - 1:0:-1]); den = 1 - np.sum(phi[k - 1, 1:k] * r[1:k])
        phi[k, k] = num / den; phi[k, 1:k] = phi[k - 1, 1:k] - phi[k, k] * phi[k - 1, k - 1:0:-1]; out.append(phi[k, k])
    return np.array(out)

LAGS = 60
x_tp = y_trainpart.values                    # training part only: hold-out excluded
check("ACF/PACF computed on training part only (hold-out excluded)", len(x_tp) == 900 and y_trainpart.index.max() < cut)
a, p = acf_np(x_tp, LAGS), pacf_np(x_tp, LAGS); band = 1.96 / np.sqrt(len(x_tp))
fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
for ax, v, col, ttl in zip(axes, (a, p), (BLUE, ORANGE), ("ACF of y(t), training part", "PACF of y(t), training part")):
    ax.vlines(range(1, LAGS + 1), 0, v[1:], color=col); ax.axhline(0, color=INK, lw=.8)
    ax.axhspan(-band, band, color=GREY, alpha=.25); ax.set_xlabel("Lag (days)"); ax.set_title(ttl)
    for k in (7, 14, 21, 28, 42, 56): ax.axvline(k, color=GREY, ls=":", lw=.6)
plt.tight_layout(); plt.show()
top = sorted(range(1, LAGS + 1), key=lambda k: -abs(a[k]))[:6]
print("Strongest ACF lags:", ", ".join(f"{k} ({a[k]:+.2f})" for k in top), f"| 95% band ±{band:.3f}")

## 8. Scaling and sequence construction
**Scaling.** `MinMaxScaler((-1, 1))` is **fitted on the fit part only**; validation and hold-out values are only *transformed* with the stored min/max. (The PDF asks for the (−1, 1) range.)

**Sliding window — simple example.** Take the toy series `[10, 20, 30, 40, 50, 60]` and a window of 3:

| input X (3 past days) | target y (next day) |
|---|---|
| 10, 20, 30 | 40 |
| 20, 30, 40 | 50 |
| 30, 40, 50 | 60 |

For our data, a target on date *t* uses the 28 values from *t−28* to *t−1*: the input never contains day *t* or anything later. Input shape is `(samples, 28, 1)` (28 time steps, 1 feature); target shape is `(samples,)`.

In [ ]:
scaler = MinMaxScaler(feature_range=(-1, 1))
scaler.fit(y_fit.values.reshape(-1, 1))                         # <- fit on the fit part ONLY
y_s = pd.Series(scaler.transform(y.values.reshape(-1, 1)).ravel(), index=y.index)   # transform only (no refit)
inverse = lambda a: scaler.inverse_transform(np.asarray(a, float).reshape(-1, 1)).ravel()

print(f"Scaler fitted on {len(y_fit)} fit days: min {scaler.data_min_[0]:,.1f}, max {scaler.data_max_[0]:,.1f}")
check("scaler fitted on the fit part only (min/max equal fit-part min/max)", np.isclose(scaler.data_min_[0], y_fit.min()) and np.isclose(scaler.data_max_[0], y_fit.max()))
check("scaler statistics not influenced by validation/hold-out", not (np.isclose(scaler.data_max_[0], y_trainpart.max()) and y_val.max() > y_fit.max()) and not (np.isclose(scaler.data_max_[0], y.max()) and y_ho.max() > y_fit.max()))

def make_sequences(series_s, target_dates, window):
    # For each target date t: X = scaled values of t-window ... t-1.
    pos = series_s.index.get_indexer(target_dates)
    assert (pos >= window).all(), "not enough history for the requested window"
    X = np.stack([series_s.values[p - window:p] for p in pos])[..., None].astype("float32")
    t = series_s.values[pos].astype("float32")
    return {"X": X, "y": t, "target_dates": pd.DatetimeIndex(target_dates),
            "first_input": series_s.index[pos - window], "last_input": series_s.index[pos - 1]}

def verify_sequences(seq, series_s, window, name):
    n = len(seq["y"])
    ok_shape = seq["X"].shape == (n, window, 1) and seq["y"].shape == (n,)
    ok_len = all((series_s.index.get_indexer(seq["last_input"]) - series_s.index.get_indexer(seq["first_input"]) + 1) == window)
    ok_noleak = bool((seq["last_input"] < seq["target_dates"]).all()) and bool(((seq["target_dates"] - seq["last_input"]) == pd.Timedelta(days=1)).all())
    ok_vals = all(np.allclose(seq["X"][i, :, 0], series_s[seq["first_input"][i]:seq["last_input"][i]].values) and np.isclose(seq["y"][i], series_s[seq["target_dates"][i]]) for i in range(n))
    check(f"{name}: shapes (n,{window},1)/(n,)", ok_shape)
    check(f"{name}: every input has exactly {window} days", ok_len)
    check(f"{name}: last input date is the day before the target (no future data)", ok_noleak)
    check(f"{name}: X/y values match the scaled series at the stated dates", ok_vals)

# toy demonstration of the construction
toy = pd.Series([10., 20, 30, 40, 50, 60], index=pd.date_range("2000-01-01", periods=6))
ts_ = make_sequences(toy, toy.index[3:], 3)
print("Toy example X:\n", ts_["X"][..., 0], "\nToy example y:", ts_["y"])

## 9. Train and validation sequence preparation
* **Training targets:** every fit-part day that has 28 days of history (first 28 fit days are only used as inputs).
* **Validation targets:** every validation day; their inputs may reach back into the fit part (that is past data, available at prediction time) but never forward.

In [ ]:
def prepare(window):
    tr = make_sequences(y_s, y_fit.index[window:], window)
    va = make_sequences(y_s, y_val.index, window)
    verify_sequences(tr, y_s, window, f"train (window {window})")
    verify_sequences(va, y_s, window, f"validation (window {window})")
    check(f"window {window}: all training/validation inputs and targets end before the hold-out", tr["target_dates"].max() < cut and va["target_dates"].max() < cut and va["last_input"].max() < cut)
    check(f"window {window}: training targets precede validation targets", tr["target_dates"].max() < va["target_dates"].min())
    return tr, va

seq_tr, seq_va = prepare(WINDOW)
print(f"Train:      X {seq_tr['X'].shape}, y {seq_tr['y'].shape}, targets {seq_tr['target_dates'][0].date()} -> {seq_tr['target_dates'][-1].date()}")
print(f"Validation: X {seq_va['X'].shape}, y {seq_va['y'].shape}, targets {seq_va['target_dates'][0].date()} -> {seq_va['target_dates'][-1].date()}")

## 10. LSTM architecture
| Item | Choice | Reason |
|---|---|---|
| Input shape | `(28, 1)` per sample | 28 days × 1 feature (scaled y) |
| Output shape | `(1,)` | next day's scaled y |
| LSTM layers | **1** (limit is 2) | simple, CPU-friendly |
| LSTM units | 32 | ~4.4 k parameters for ~800 training windows; larger models would over-fit |
| Dropout | **none** | the model is tiny; dropout is only added if the validation curve shows over-fitting (see section 12) |
| Output | `Dense(1)` linear | regression |
| Loss / optimizer | MSE / Adam (lr 1e-3) | MSE on scaled data is the usual regression loss and smooth to optimise |
| Batch size / max epochs | 32 / 100 | small and fast |
| Early stopping | monitor `val_loss`, patience 12, `restore_best_weights=True` | stops when validation stops improving and keeps the best weights |

Training windows are shuffled *within the training set only* (they are independent samples; each already contains its own past). Validation and hold-out are never shuffled.

In [ ]:
def build_model(window, units=LSTM_UNITS, dropout=DROPOUT):
    tf.keras.backend.clear_session(); tf.keras.utils.set_random_seed(SEED)
    layers = [tf.keras.layers.Input(shape=(window, 1)), tf.keras.layers.LSTM(units)]
    if dropout > 0: layers.append(tf.keras.layers.Dropout(dropout))
    layers.append(tf.keras.layers.Dense(1))
    m = tf.keras.Sequential(layers)
    m.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss="mse")
    n_lstm = sum(isinstance(l, tf.keras.layers.LSTM) for l in m.layers)
    assert n_lstm <= 2, "project limit: at most two LSTM layers"
    return m

model = build_model(WINDOW)
model.summary()
n_lstm_layers = sum(isinstance(l, tf.keras.layers.LSTM) for l in model.layers)
check("LSTM layers <= 2 (project limit)", n_lstm_layers <= 2)
print(f"LSTM layers: {n_lstm_layers} | units: {LSTM_UNITS} | dropout: {DROPOUT} | parameters: {model.count_params():,}")

## 11. Model training (fit part only; early stopping on the validation part)

In [ ]:
def train_model(window, tr, va):
    m = build_model(window)
    es = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=PATIENCE, restore_best_weights=True)
    t0 = time.time()
    hist = m.fit(tr["X"], tr["y"], validation_data=(va["X"], va["y"]), epochs=MAX_EPOCHS, batch_size=BATCH_SIZE,
                 shuffle=True, callbacks=[es], verbose=0)
    h = hist.history
    info = {"epochs_run": len(h["loss"]), "best_epoch": int(np.argmin(h["val_loss"])) + 1, "best_val_loss": float(np.min(h["val_loss"])),
            "train_loss_at_best": float(h["loss"][int(np.argmin(h["val_loss"]))]), "seconds": time.time() - t0, "monitor": es.monitor}
    return m, h, info

model_28, hist_28, info_28 = train_model(WINDOW, seq_tr, seq_va)
print({k: (round(v, 6) if isinstance(v, float) else v) for k, v in info_28.items()})
check("early stopping monitored the validation loss", info_28["monitor"] == "val_loss")
check("training finished within the epoch limit", info_28["epochs_run"] <= MAX_EPOCHS)

## 12. Training-history plots

In [ ]:
def plot_history(h, info, title):
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.plot(range(1, len(h["loss"]) + 1), h["loss"], color=BLUE, label="training loss")
    ax.plot(range(1, len(h["val_loss"]) + 1), h["val_loss"], color=ORANGE, label="validation loss")
    ax.axvline(info["best_epoch"], color=GREY, ls=":", label=f"best epoch ({info['best_epoch']})")
    ax.set_xlabel("Epoch"); ax.set_ylabel("MSE (scaled units)"); ax.set_yscale("log"); ax.set_title(title); ax.legend(frameon=False)
    plt.tight_layout(); plt.show()
plot_history(hist_28, info_28, f"Training history — window {WINDOW}")
gap = info_28["best_val_loss"] / info_28["train_loss_at_best"]
display(Markdown(f"At the best epoch the validation loss is **{gap:.2f}×** the training loss. "
                 f"{'The gap is small, so dropout is not justified.' if gap < 3 else 'The gap is large; dropout could be tried, but only judged on validation data.'}"))

## 13. Validation results
Metrics (all on the original EUR scale):
* **RMSPE** = √mean(((actual − pred)/actual)²) — the same definition as the Time Series and ML notebooks; penalises large *relative* errors. It divides by the actual value, so it breaks if an actual is 0. Here y(t) is a mean over trading stores with positive sales, so every actual is strictly positive (checked); no epsilon, filtering or other change is applied, and the function raises an error if a non-positive actual ever appears.
* **RMSE** = √mean((actual − pred)²) — typical error in EUR, large errors weigh more.
* **MAE** = mean(|actual − pred|) — average error in EUR, easy to read.

The naive benchmarks from the Time Series notebook are re-created below (same definitions) and scored on the same validation days.

In [ ]:
def metrics(actual, pred):
    actual, pred = np.asarray(actual, float), np.asarray(pred, float)
    if not (actual > 0).all(): raise ValueError("RMSPE undefined: non-positive actual value")
    return {"RMSPE": float(np.sqrt(np.mean(((actual - pred) / actual) ** 2))),
            "RMSE": float(np.sqrt(np.mean((actual - pred) ** 2))),
            "MAE": float(np.mean(np.abs(actual - pred)))}

# --- naive benchmarks, identical to 03_Time_Series_Analysis.ipynb section 10 ---
one_step = {"same weekday last week (lag 7)": y.shift(7), "same weekday 2 weeks ago (lag 14)": y.shift(14),
            "same weekday 4 weeks ago (lag 28)": y.shift(28), "mean of lags 14 and 28": (y.shift(14) + y.shift(28)) / 2,
            "mean of lags 7, 14, 21, 28": (y.shift(7) + y.shift(14) + y.shift(21) + y.shift(28)) / 4}

def repeat_block(series, origin, block, horizon=HOLDOUT_DAYS):
    hist = series[:origin].iloc[-block:].values           # only data up to the origin
    return np.array([hist[i % block] for i in range(horizon)])

def predict_dates(m, series_s, dates, window):
    seq = make_sequences(series_s, dates, window)
    assert not np.isnan(seq["X"]).any()
    return inverse(m.predict(seq["X"], batch_size=256, verbose=0)), seq   # no shuffling, order = date order

pred_val, _ = predict_dates(model_28, y_s, y_val.index, WINDOW)
check("validation predictions finite and one per validation day", np.isfinite(pred_val).all() and len(pred_val) == len(y_val) == VAL_DAYS)
val_rows = [{"model": f"LSTM (window {WINDOW})", **metrics(y_val, pred_val)}] + \
           [{"model": "naive: " + k, **metrics(y_val, v[y_val.index])} for k, v in one_step.items()]
val_table = pd.DataFrame(val_rows); display(val_table.round(4))

## 14. Final hold-out evaluation
From here on the 42 hold-out days are used **only for scoring**, never for tuning or model selection.

* The **28-day LSTM is the primary model and the final evaluation.** It was designated as primary *in advance* (window chosen from the Time Series notebook's ACF analysis, fixed in section 3) and its epoch count was chosen by **validation loss** only.
* The **42-day LSTM (section 18) is a pre-planned sensitivity evaluation.** It is also scored on the hold-out for information, but that score is **not** used to choose, replace or re-tune the primary model; the primary model stays the 28-day model whatever the sensitivity result is.
* The hold-out is therefore scored for both models, but **no decision of any kind depends on those scores**: no architecture, hyper-parameter, epoch count or window was chosen with them, and no model is re-trained afterwards.

In [ ]:
results = []     # comparison table rows
def add_result(model_name, mode, horizon, actual, pred):
    role = "benchmark" if model_name.startswith("naive") else ("sensitivity (pre-planned)" if "sensitivity" in model_name else "PRIMARY (final)")
    results.append({"model / benchmark": model_name, "role": role, "evaluation mode": mode, "forecast horizon": horizon, **metrics(actual, pred)})

selection_snapshot = {"window": WINDOW, "units": LSTM_UNITS, "dropout": DROPOUT, "best_epoch": info_28["best_epoch"], "lstm_layers": n_lstm_layers}
ROLL, FIXED = "A: rolling one-step-ahead", "B: fixed-origin multi-step"
HZ_ROLL, HZ_FIXED = "1 day (42 dates)", f"42 days from {origin.date()}"
print("Hold-out:", y_ho.index[0].date(), "->", y_ho.index[-1].date(), f"({len(y_ho)} days) | primary window {WINDOW}")

## 15. Rolling one-step-ahead evaluation (Mode A)
For each hold-out date *t* the input is the **actual** y of the 28 days before *t*. Actual values of earlier hold-out days are therefore used as they "become available" (day *t−1* is known when predicting *t*) — the same information the one-step naive forecasts use (`y.shift(k)`). The model is not re-trained or updated; only its input window moves.

In [ ]:
pred_roll, seq_ho = predict_dates(model_28, y_s, y_ho.index, WINDOW)
verify_sequences(seq_ho, y_s, WINDOW, f"hold-out rolling (window {WINDOW})")
check("rolling: exactly 42 hold-out predictions, finite", len(pred_roll) == HOLDOUT_DAYS and np.isfinite(pred_roll).all())
check("rolling: first hold-out input window ends on the last training day (origin)", seq_ho["last_input"][0] == origin)
check("rolling: hold-out inputs use only actual data strictly before each target date", (seq_ho["last_input"] < seq_ho["target_dates"]).all())

m_roll = metrics(y_ho, pred_roll)
add_result(f"LSTM (window {WINDOW})", ROLL, HZ_ROLL, y_ho, pred_roll)
for k, v in one_step.items():
    add_result("naive: " + k, ROLL, HZ_ROLL, y_ho, v[y_ho.index])
print({k: round(v, 4) for k, v in m_roll.items()})

## 16. Fixed-origin 42-day forecast (Mode B)
Origin = last training day (2015-06-19). The model predicts day 1, appends **its own prediction** to the window, predicts day 2, and so on for 42 steps. **No hold-out actual enters the input.** Errors can accumulate, so this is much harder than Mode A.

In [ ]:
def recursive_forecast(m, series_s, origin, window, horizon=HOLDOUT_DAYS):
    hist = series_s[:origin]
    assert hist.index[-1] == origin, "origin not found in series"
    buf = list(hist.values[-window:])                       # actual scaled values up to the origin ONLY
    out = []
    for _ in range(horizon):
        x = np.asarray(buf[-window:], dtype="float32").reshape(1, window, 1)
        p = float(np.asarray(m(x, training=False)).ravel()[0])
        out.append(p); buf.append(p)                         # feed the prediction, never an actual
    return inverse(out)

pred_fixed = recursive_forecast(model_28, y_s, origin, WINDOW)
check("fixed-origin: 42 finite predictions", len(pred_fixed) == HOLDOUT_DAYS and np.isfinite(pred_fixed).all())
check("fixed-origin: origin is the last training day, first forecast day is the first hold-out day", origin == y_trainpart.index[-1] and origin + pd.Timedelta(days=1) == y_ho.index[0])

# Leakage test: forecasts must be unchanged if all hold-out actuals are replaced by garbage
y_s_garbage = y_s.copy(); y_s_garbage[y_s_garbage.index >= cut] = 123.0
check("fixed-origin: forecast does not depend on hold-out actuals (identical when they are corrupted)", np.allclose(pred_fixed, recursive_forecast(model_28, y_s_garbage, origin, WINDOW)))

add_result(f"LSTM (window {WINDOW})", FIXED, HZ_FIXED, y_ho, pred_fixed)
fixed_naive = {f"repeat the last {b} days": repeat_block(y, origin, b) for b in (7, 14, 28)}
for k, v in fixed_naive.items():
    check(f"benchmark '{k}': same origin ({origin.date()}) and horizon (42) as the LSTM", len(v) == len(pred_fixed))
    add_result("naive: " + k, FIXED, HZ_FIXED, y_ho, v)
print({k: round(v, 4) for k, v in metrics(y_ho, pred_fixed).items()})

## 17. Matching naive benchmark comparison
* The table below is a first version (primary model + naive benchmarks); it is **rebuilt in sections 18 and 21** once the sensitivity rows exist.
* Mode A (rolling) is compared **only** with the one-step naive forecasts; Mode B (fixed-origin) **only** with the "repeat the last block" forecasts. Both use the same dates and the same RMSPE definition as `03_Time_Series_Analysis.ipynb`, so these are exact comparisons.
* These RMSPE values are for the **chain-average daily series** and must not be compared with store-level RMSPE in the ML notebook — a different, much harder task.

In [ ]:
comp = pd.DataFrame(results)
display(comp.round(4))

def verdict(mode, lstm_name):
    sub = comp[comp["evaluation mode"] == mode]
    l = sub[sub["model / benchmark"] == lstm_name].iloc[0]; nv = sub[sub["model / benchmark"].str.startswith("naive")].sort_values("RMSPE")
    best, worst = nv.iloc[0], nv.iloc[-1]
    rel = (l.RMSPE / best.RMSPE - 1) * 100
    return (f"**{mode}** — LSTM RMSPE **{l.RMSPE:.4f}** vs best naive **{best['model / benchmark'][7:]}** {best.RMSPE:.4f} "
            f"({'LSTM is ' + format(abs(rel), '.0f') + '% better' if rel < 0 else 'LSTM is ' + format(rel, '.0f') + '% worse'}) and worst naive {worst.RMSPE:.4f}. "
            f"LSTM RMSE {l.RMSE:,.0f} / MAE {l.MAE:,.0f} EUR vs best naive RMSE {best.RMSE:,.0f} / MAE {best.MAE:,.0f} EUR.")
display(Markdown(verdict(ROLL, f"LSTM (window {WINDOW})"))); display(Markdown(verdict(FIXED, f"LSTM (window {WINDOW})")))

## 18. 28-day versus 42-day window sensitivity experiment
Same architecture and settings, only the input window changes (42 days). Trained and early-stopped on the same validation days. This is a **pre-planned sensitivity evaluation**, not a selection step: the 28-day model stays the primary final model **whatever** this experiment shows, and the hold-out is not used to pick a window. The 42-day model is trained and early-stopped on validation data only. If anything fails below, the reason is printed and nothing is invented.

In [ ]:
sens_done, sens_error = False, None
try:
    seq_tr42, seq_va42 = prepare(WINDOW_SENS)
    model_42, hist_42, info_42 = train_model(WINDOW_SENS, seq_tr42, seq_va42)
    pv42, _ = predict_dates(model_42, y_s, y_val.index, WINDOW_SENS)
    pr42, seq_ho42 = predict_dates(model_42, y_s, y_ho.index, WINDOW_SENS)
    verify_sequences(seq_ho42, y_s, WINDOW_SENS, f"hold-out rolling (window {WINDOW_SENS})")
    pf42 = recursive_forecast(model_42, y_s, origin, WINDOW_SENS)
    check("window 42: predictions finite and correctly sized", np.isfinite(pv42).all() and np.isfinite(pr42).all() and np.isfinite(pf42).all() and len(pr42) == len(pf42) == HOLDOUT_DAYS)
    plot_history(hist_42, info_42, f"Training history — window {WINDOW_SENS} (sensitivity)")
    sens_done = True
except Exception as e:
    sens_error = repr(e)
    print("42-day sensitivity experiment NOT completed. Reason:", sens_error)

if sens_done:
    sens = pd.DataFrame([
        {"window": WINDOW, "role": "primary", "best epoch": info_28["best_epoch"], "val RMSPE": metrics(y_val, pred_val)["RMSPE"],
         "hold-out RMSPE (rolling A)": m_roll["RMSPE"], "hold-out RMSPE (fixed-origin B)": metrics(y_ho, pred_fixed)["RMSPE"], "hold-out RMSE (A)": m_roll["RMSE"], "hold-out MAE (A)": m_roll["MAE"]},
        {"window": WINDOW_SENS, "role": "sensitivity", "best epoch": info_42["best_epoch"], "val RMSPE": metrics(y_val, pv42)["RMSPE"],
         "hold-out RMSPE (rolling A)": metrics(y_ho, pr42)["RMSPE"], "hold-out RMSPE (fixed-origin B)": metrics(y_ho, pf42)["RMSPE"], "hold-out RMSE (A)": metrics(y_ho, pr42)["RMSE"], "hold-out MAE (A)": metrics(y_ho, pr42)["MAE"]}])
    display(sens.round(4))
    add_result(f"LSTM (window {WINDOW_SENS}, sensitivity)", ROLL, HZ_ROLL, y_ho, pr42)
    add_result(f"LSTM (window {WINDOW_SENS}, sensitivity)", FIXED, HZ_FIXED, y_ho, pf42)
    better_val = sens.iloc[1]["val RMSPE"] < sens.iloc[0]["val RMSPE"]
    comp = pd.DataFrame(results)      # REBUILD the consolidated table so the sensitivity rows are included
    print("Consolidated comparison table updated: now", len(comp), "rows (includes the 42-day sensitivity rows).")
    display(comp.round(4))
    display(Markdown(f"On **validation** data the {WINDOW_SENS}-day window is {'better' if better_val else 'not better'} than the {WINDOW}-day window. The primary model remains the {WINDOW}-day LSTM (fixed in advance); hold-out numbers of the 42-day model are shown for information only."))

## 19. Actual-versus-predicted plots

In [ ]:
best_naive_roll = comp[(comp["evaluation mode"] == ROLL) & comp["model / benchmark"].str.startswith("naive")].sort_values("RMSPE").iloc[0]["model / benchmark"][7:]
best_naive_fix = comp[(comp["evaluation mode"] == FIXED) & comp["model / benchmark"].str.startswith("naive")].sort_values("RMSPE").iloc[0]["model / benchmark"][7:]

fig, axes = plt.subplots(3, 1, figsize=(13, 11))
ax = axes[0]
ax.plot(y_val.index, y_val, color=INK, lw=2, label="Actual"); ax.plot(y_val.index, pred_val, color=BLUE, lw=1.6, label=f"LSTM (window {WINDOW}), rolling")
ax.set_title("Validation period: actual vs LSTM (one-step)"); ax.legend(frameon=False, ncol=2); ax.set_ylabel("EUR per trading store")
ax = axes[1]
ax.plot(y_ho.index, y_ho, color=INK, lw=2.2, label="Actual"); ax.plot(y_ho.index, pred_roll, color=BLUE, lw=1.8, label="LSTM rolling one-step")
ax.plot(y_ho.index, one_step[best_naive_roll][y_ho.index], color=ORANGE, lw=1.4, ls="--", label=f"best naive one-step: {best_naive_roll}")
ax.set_title("Hold-out, Mode A: rolling one-step-ahead"); ax.legend(frameon=False, ncol=3, fontsize=8, loc="lower center", bbox_to_anchor=(.5, -.32)); ax.set_ylabel("EUR per trading store")
ax = axes[2]
ax.plot(y_ho.index, y_ho, color=INK, lw=2.2, label="Actual"); ax.plot(y_ho.index, pred_fixed, color=AQUA, lw=1.8, label="LSTM fixed-origin 42 days")
ax.plot(y_ho.index, fixed_naive[best_naive_fix], color=ORANGE, lw=1.4, ls="--", label=f"best naive 42-day: {best_naive_fix}")
ax.set_title(f"Hold-out, Mode B: fixed-origin 42-day forecast from {origin.date()}"); ax.legend(frameon=False, ncol=3, fontsize=8, loc="lower center", bbox_to_anchor=(.5, -.32)); ax.set_ylabel("EUR per trading store")
for a in axes: a.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.show()

## 20. Residual / error analysis

In [ ]:
res_roll = y_ho.values - pred_roll;  res_fix = y_ho.values - pred_fixed
pe_roll = res_roll / y_ho.values * 100; pe_fix = res_fix / y_ho.values * 100

fig, axes = plt.subplots(2, 2, figsize=(13, 7.5))
axes[0, 0].bar(y_ho.index, pe_roll, color=BLUE, width=.8); axes[0, 0].axhline(0, color=INK, lw=.8); axes[0, 0].set_title("Mode A: percentage error by day"); axes[0, 0].set_ylabel("(actual − pred)/actual, %")
axes[0, 1].hist(res_roll, bins=12, color=BLUE, alpha=.85); axes[0, 1].axvline(0, color=INK, lw=.8); axes[0, 1].set_title("Mode A: residual distribution (EUR)")
axes[1, 0].plot(range(1, HOLDOUT_DAYS + 1), np.abs(pe_fix), color=AQUA, marker="o", ms=3, label="LSTM fixed-origin")
nv = (y_ho.values - fixed_naive[best_naive_fix]) / y_ho.values * 100
axes[1, 0].plot(range(1, HOLDOUT_DAYS + 1), np.abs(nv), color=ORANGE, ls="--", label=f"naive: {best_naive_fix}")
axes[1, 0].set_title("Mode B: absolute % error by forecast day"); axes[1, 0].set_xlabel("Days ahead"); axes[1, 0].legend(frameon=False)
axes[1, 1].scatter(pred_roll, y_ho.values, color=BLUE, s=22); lo, hi = min(pred_roll.min(), y_ho.min()), max(pred_roll.max(), y_ho.max())
axes[1, 1].plot([lo, hi], [lo, hi], color=INK, lw=1); axes[1, 1].set_xlabel("Predicted"); axes[1, 1].set_ylabel("Actual"); axes[1, 1].set_title("Mode A: actual vs predicted")
for a in axes.ravel(): a.tick_params(axis="x", labelrotation=25 if a in (axes[0, 0],) else 0)
plt.tight_layout(); plt.show()

dn = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
by_dow = pd.DataFrame({"weekday": [dn[d] for d in y_ho.index.dayofweek], "abs % error (A)": np.abs(pe_roll), "bias % (A)": pe_roll}).groupby("weekday").mean().reindex(dn)
display(by_dow.round(2))
worst = pd.DataFrame({"date": y_ho.index.date, "actual": y_ho.values.round(0), "LSTM rolling": pred_roll.round(0), "% error": pe_roll.round(2)}).iloc[np.argsort(-np.abs(pe_roll))[:5]]
print("Five largest Mode A errors:"); display(worst)
print(f"Mode A mean bias (actual − pred): {res_roll.mean():,.0f} EUR | Mode B mean bias: {res_fix.mean():,.0f} EUR")

## 21. Results summary

In [ ]:
# ---- final consolidated table: rebuilt from ALL completed results (primary, benchmarks, sensitivity) ----
comp = pd.DataFrame(results)
check("final comparison table contains every completed result", len(comp) == len(results) and (not sens_done or comp["model / benchmark"].str.contains("sensitivity").sum() == 2))
# ---- final consolidated validation checks ----
check("no model selection on hold-out: configuration unchanged since validation-based training", selection_snapshot == {"window": WINDOW, "units": LSTM_UNITS, "dropout": DROPOUT, "best_epoch": info_28["best_epoch"], "lstm_layers": n_lstm_layers} and WINDOW == 28)
check("hold-out metrics all finite", all(np.isfinite(comp[["RMSPE", "RMSE", "MAE"]].values.ravel())))
check("Mode A and Mode B kept in separate rows (never mixed)", set(comp["evaluation mode"]) == {ROLL, FIXED})
check("number of hold-out predictions = 42 in both modes", len(pred_roll) == len(pred_fixed) == len(y_ho) == 42)
display(pd.DataFrame(checks)); print(f"{sum(c['passed'] for c in checks)}/{len(checks)} checks passed")

final = comp.copy(); final["RMSE"] = final["RMSE"].round(0); final["MAE"] = final["MAE"].round(0); final["RMSPE"] = final["RMSPE"].round(4)
display(final)
print("Final consolidated comparison (all completed models: primary, pre-planned sensitivity, benchmarks):")
display(Markdown(verdict(ROLL, f"LSTM (window {WINDOW})"))); display(Markdown(verdict(FIXED, f"LSTM (window {WINDOW})")))

## 22. Limitations and conclusion

**Limitations**
1. **Univariate input.** The LSTM sees only past values of y(t). The Time Series notebook showed that the promotion calendar and holidays are known in advance and explain much of the two-week rhythm; adding them as extra input channels is the natural next step.
2. **Easy hold-out window.** The hold-out (20 Jun – 31 Jul 2015) contains no public holiday and no Christmas period, so scores are optimistic compared with the 42 days before it, where the naive forecasts did much worse (see the Time Series notebook, section 10.1). Yearly seasonality is not tested.
3. **Short history, short validation.** About 2.6 years of data and a 56-day validation window make early stopping noisy.
4. **Single seed.** Results come from one seeded run; a different seed can change the numbers slightly.
5. **Recursive forecasting** (Mode B) lets errors accumulate over 42 steps and tends to drift towards an average weekly pattern.
6. **Chain-average target.** The series averages over whichever stores trade that day (store-mix effect on Sundays/holidays). RMSPE values here are **not comparable** with the store-level RMSPE of the ML notebook.
7. **Scaler range.** Values outside the fit-part range (−1…1) are possible in new periods; the LSTM may extrapolate poorly there.

**Conclusion.** The notebook builds a one-layer, 32-unit LSTM on 28-day windows with chronological train / validation / hold-out splitting and no leakage, and evaluates it in two clearly separated modes against the matching naive benchmarks. Read the numbers in sections 17 and 21: they are produced by the code above and are the only results this notebook claims.